# Day 5 — Synthetic Test Data Generator (Open‑Source)  
**Business challenge:** Build a product that creates realistic, diverse, edge‑case‑aware **synthetic datasets** with an open‑source model and a **Gradio** UI.

### Highlights
- Fully open‑source LLM (default: `microsoft/Phi-3-mini-4k-instruct`) with 4‑bit quantization when possible
- Generate **JSON**, **CSV**, or **Markdown tables**
- Controls for **record count**, **diversity**, and **edge cases**
- Simple **data quality checks**

## 0) Installs

In [ ]:
# If on Colab with GPU, install the CUDA-enabled PyTorch build separately if desired.
# !pip install -q --upgrade torch torchvision torchaudio

!pip install -q -U transformers accelerate bitsandbytes sentencepiece gradio pandas python-dateutil

## 1) Imports & device

In [ ]:
import os, json, re, ast, tempfile, datetime
from typing import List, Dict, Any
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import gradio as gr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 2) Load an open‑source chat model (4‑bit when possible)

In [ ]:
DEFAULT_MODEL = "microsoft/Phi-3-mini-4k-instruct"
MODEL_OPTIONS = [
    "microsoft/Phi-3-mini-4k-instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "HuggingFaceH4/zephyr-7b-beta",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]

def load_model(model_name: str):
    print(f"Loading: {model_name}")
    kwargs = {}
    if DEVICE == "cuda":
        try:
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            kwargs["device_map"] = "auto"
            kwargs["torch_dtype"] = torch.bfloat16
        except Exception as e:
            print("4-bit quantization failed, falling back:", e)
            kwargs["device_map"] = "auto"
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    return tok, mdl

tokenizer, model = load_model(DEFAULT_MODEL)

## 3) Prompting for structured output
We instruct the model to return a **strict JSON array** of objects. You supply a **schema description** and we build a guiding prompt.

In [ ]:
SYSTEM = (
    "You are a meticulous synthetic data generator that outputs only JSON arrays of objects "
    "with realistic, diverse, and edge‑case‑aware values. "
    "Do not include any commentary, explanations, or code fences—return raw JSON only."
)

def build_generation_prompt(schema_desc: str, n_rows: int, style: str, anomaly_ratio: float):
    now = datetime.datetime.now().strftime("%Y-%m-%d")
    user = f"""
    Create a synthetic dataset **in JSON** with exactly {n_rows} rows and realistic values.
    {schema_desc}

    Requirements:
    - Return a JSON array of objects only (no code fences, no commentary).
    - Include diverse, internationalized values where reasonable.
    - Include dates formatted ISO-8601 (YYYY-MM-DD) when dates are present (today is {now}).
    - Style: {style}.
    - Include edge cases: {int(100*anomaly_ratio)}% of records should contain edge conditions (e.g., nulls, long strings, unusual unicode, boundary values, unexpected but valid categories).
    """
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
    ]
    return messages

def generate_raw_json(schema_desc: str, n_rows: int, style: str, anomaly_ratio: float, temperature: float, top_p: float, max_new_tokens: int = 3000):
    messages = build_generation_prompt(schema_desc, n_rows, style, anomaly_ratio)
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    if DEVICE == "cuda":
        input_ids = input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0, input_ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    # Strip code fences if the model added them
    text = re.sub(r'^```(?:json)?\n|\n```$', '', text.strip(), flags=re.IGNORECASE)
    return text

## 4) Parsing & export helpers

In [ ]:
def parse_json_array(raw_text: str):
    try:
        data = json.loads(raw_text)
        if not isinstance(data, list):
            raise ValueError("JSON root is not a list.")
        return data
    except Exception:
        # try a crude repair: locate the first '[' and last ']'
        start = raw_text.find('[')
        end = raw_text.rfind(']')
        if start >= 0 and end > start:
            try:
                return json.loads(raw_text[start:end+1])
            except Exception as e:
                raise e
        raise

def to_dataframe(records: List[Dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame.from_records(records)

def dataframe_to_csv(df: pd.DataFrame) -> str:
    tmp = tempfile.mkstemp(suffix=".csv")[1]
    df.to_csv(tmp, index=False)
    return tmp

def dataframe_to_json(df: pd.DataFrame) -> str:
    tmp = tempfile.mkstemp(suffix=".json")[1]
    df.to_json(tmp, orient="records", force_ascii=False, indent=2)
    return tmp

def dataframe_to_markdown(df: pd.DataFrame, max_rows: int = 50) -> str:
    display_df = df.head(max_rows)
    return display_df.to_markdown(index=False)

## 5) Gradio UI

In [ ]:
def ensure_model(selected: str):
    global tokenizer, model
    if selected and selected != getattr(model, "name_or_path", None):
        tokenizer, model = load_model(selected)
    return selected

EXAMPLES = [
    "Schema: customer support tickets with fields: ticket_id (string), created_date (date), customer_name (string), email (string), product (category), issue_summary (string), priority (low|medium|high|urgent), assigned_agent (string), sla_due_date (date), status (open|pending|resolved|closed).",
    "Schema: e-commerce orders: order_id (string), order_date (date), customer_id (string), ship_country (string), items (array of {sku, name, qty, unit_price}), payment_method (enum), discount_code (nullable string), total (number).",
    "Schema: job postings: job_id (string), title (string), department (string), location (string), salary_range (string), employment_type (enum), description (string), posting_date (date), application_deadline (date).",
    "Schema: financial transactions: tx_id (string), timestamp (date), account_id (string), amount (number), currency (string), merchant (string), mcc (string), channel (enum), risk_score (0-1 float).",
]

def synthesize(schema_desc, n_rows, style, anomaly_ratio, model_name, temperature, top_p, out_format):
    ensure_model(model_name)
    raw = generate_raw_json(schema_desc, int(n_rows), style, float(anomaly_ratio), float(temperature), float(top_p))
    try:
        records = parse_json_array(raw)
    except Exception as e:
        return f"⚠️ Could not parse JSON. Raw text shown below for debugging.\n\n````\n{raw}\n````", None, None, None

    df = to_dataframe(records)
    md_preview = dataframe_to_markdown(df, max_rows=50)
    csv_file = dataframe_to_csv(df)
    json_file = dataframe_to_json(df)
    out_file = csv_file if out_format == "CSV" else json_file
    return md_preview, csv_file, json_file, out_file

with gr.Blocks(title="Synthetic Test Data Generator") as gen_app:
    gr.Markdown("## Synthetic Test Data Generator (Open‑Source)")
    with gr.Row():
        with gr.Column():
            schema = gr.Textbox(label="Describe your schema", lines=8, value=EXAMPLES[0])
            n_rows = gr.Slider(1, 500, step=1, value=100, label="Rows")
            style = gr.Dropdown(
                label="Style", value="realistic",
                choices=["realistic", "noisy", "edge‑cases‑heavy", "balanced"]
            )
            anomaly_ratio = gr.Slider(0.0, 0.5, step=0.05, value=0.1, label="Edge case ratio")
            model_name = gr.Dropdown(label="Open‑source model", value=DEFAULT_MODEL, choices=MODEL_OPTIONS)
            temperature = gr.Slider(0.0, 1.5, value=0.8, step=0.05, label="Temperature")
            top_p = gr.Slider(0.1, 1.0, value=0.95, step=0.05, label="top_p")
            out_format = gr.Radio(label="Primary download format", choices=["CSV", "JSON"], value="CSV")
            btn = gr.Button("Generate Dataset", variant="primary")
        with gr.Column():
            preview = gr.Markdown(label="Preview (first rows)")
            csv_file = gr.File(label="CSV download")
            json_file = gr.File(label="JSON download")
            final_file = gr.File(label="Primary download")

    btn.click(
        fn=synthesize,
        inputs=[schema, n_rows, style, anomaly_ratio, model_name, temperature, top_p, out_format],
        outputs=[preview, csv_file, json_file, final_file],
    )

print("✅ App ready. In Colab/Jupyter, run gen_app.launch(share=True) to open the UI.")

### Launch the app (Colab or local Jupyter)

In [ ]:
# Uncomment to run in-notebook.
# gen_app.launch(share=True)